# Modeling - iFood Challenge

## 1) Import libraries

In [0]:
%pip install lightgbm
%pip install shap

In [0]:
import pandas as pd
import numpy as np

from lightgbm import LGBMClassifier

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)

import shap

## 2) Read dataset

In [0]:
DATA_PATH = "/Volumes/workspace/default/processed"
df = pd.read_parquet(DATA_PATH)
print(df.shape)
df.head()

In [0]:
df["converted"].value_counts(normalize=True)

In [0]:
df.isnull().sum().sort_values(ascending=False).head(10)

## 3) Feature preprocessing and target split

In [0]:
TARGET = "converted"

DROP_COLS = [
    "converted",
    "account_id",
    "offer_id",
    "t_completed",   # post-event info
]

X = df.drop(columns=DROP_COLS)
y = df[TARGET]


In [0]:
X["cohort"] = pd.to_datetime(X["cohort"], format="%Y-%m")

X = X.sort_values("cohort")
y = y.loc[X.index]


In [0]:

X["cohort_month"] = X["cohort"].dt.month
X.display()

In [0]:
categorical_features = [
    "gender",
    "offer_type"
]
for col in categorical_features:
    X[col] = X[col].astype("category")


## 4) Temporal Cross-Validation (Rolling Window)
Temporal cross-validation aims to avoid data leakage, always train with the past and validate with the future. In the Rolling Window method, we train with the beginning of the time series and validate with the next period then we increase the size of the train and validate with the next period.

In [0]:
unique_months = X["cohort"].sort_values().unique()

N_FOLDS = 5
fold_size = len(unique_months) // (N_FOLDS + 1)

In [0]:
def temporal_cv_splits(months, n_folds, fold_size):
    splits = []
    for i in range(n_folds):
        train_end = (i + 1) * fold_size
        val_end = train_end + fold_size

        train_months = months[:train_end]
        val_months = months[train_end:val_end]

        splits.append((train_months, val_months))
    return splits


In [0]:
cv_splits = temporal_cv_splits(unique_months, N_FOLDS, fold_size)

In [0]:
print("N_FOLDS = ", N_FOLDS)
print("fold_size = ", fold_size)

## 5) Model training - Cross validation
Check model stability in time and performance variation.

In [0]:
model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [0]:
cv_results = []

for fold, (train_months, val_months) in enumerate(cv_splits, 1):
    print(f"\n🔹 Fold {fold}")

    train_idx = X["cohort"].isin(train_months)
    val_idx = X["cohort"].isin(val_months)

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    # REMOVE cohort before feeding into LightGBM
    X_train = X_train.drop(columns=["cohort"])
    X_val   = X_val.drop(columns=["cohort"])

    model.fit(
        X_train,
        y_train,
        categorical_feature=categorical_features
    )

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1]

    metrics = {
        "fold": fold,
        "auc": roc_auc_score(y_val, y_prob),
        "accuracy": accuracy_score(y_val, y_pred),
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred)
    }

    cv_results.append(metrics)

    print(metrics)


In [0]:
X_train.dtypes

In [0]:
cv_df = pd.DataFrame(cv_results)
cv_df

In [0]:
cv_df.mean()

## 6) Model training - Train/test split
Train with past data and test with future data (similar behavior of real life)

In [0]:
train_months = unique_months[:-fold_size]
test_months = unique_months[-fold_size:]

In [0]:
X_train = X[X["cohort"].isin(train_months)]
y_train = y.loc[X_train.index]

X_test = X[X["cohort"].isin(test_months)]
y_test = y.loc[X_test.index]


In [0]:
final_model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
X_train = X_train.drop(columns=["cohort"])
final_model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_features
)

In [0]:
X_train.dtypes

In [0]:
X_test = X_test.drop(columns=["cohort"])
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

In [0]:
print("AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

In [0]:
confusion_matrix(y_test, y_pred)

## 7) Shap values

In [0]:
# Create SHAP explainer for LightGBM
explainer = shap.TreeExplainer(model)

# Compute SHAP values for the validation fold
shap_values = explainer.shap_values(X_val)

In [0]:
shap.summary_plot(shap_values, X_val)

In [0]:
shap.summary_plot(shap_values, X_val, plot_type="bar")

## 8) Future work
To improve the model quality and analysis, some additional steps could be done as:
- More feature engineering;
- Outliers removal;
- Feature selection;
- Model hyperparameter tunning;
- Adjust model weight to deal with unbalanced class;

In [0]:
#!pip freeze > requirements.txt